# 5.15 The EM algorithm

When labels are hidden, alternate between soft completion and ordinary fitting.

EM climbs the observed latent-variable likelihood by alternating responsibilities and parameter updates. We implement a real Bernoulli-mixture EM and stress it with overlapping, missing-label, and high-dimensional mixtures.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 515
rng = np.random.default_rng(SEED)


def logsumexp(a, axis=None, keepdims=False):
    a = np.asarray(a, dtype=float)
    m = np.max(a, axis=axis, keepdims=True)
    value = m + np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True))
    if not keepdims:
        value = np.squeeze(value, axis=axis)
    return value


def em_latent_coin(sequences, K=2, max_iter=50, init_theta=None, init_pi=None):
    X = np.asarray(sequences, dtype=float)
    n, d = X.shape
    if init_theta is None:
        init_theta = np.linspace(0.25, 0.75, K)
    if init_pi is None:
        init_pi = np.ones(K) / K
    theta = np.asarray(init_theta, dtype=float).copy()
    pi = np.asarray(init_pi, dtype=float).copy()
    history = []
    responsibilities = np.zeros((n, K))
    for _ in range(max_iter):
        log_prob = np.zeros((n, K))
        heads = X.sum(axis=1)
        tails = d - heads
        for k in range(K):
            log_prob[:, k] = np.log(pi[k]) + heads * np.log(theta[k]) + tails * np.log(1.0 - theta[k])
        log_norm = logsumexp(log_prob, axis=1, keepdims=True)
        responsibilities = np.exp(log_prob - log_norm)
        ll = float(np.sum(log_norm))
        history.append(ll)
        Nk = responsibilities.sum(axis=0)
        pi = Nk / n
        theta = (responsibilities.T @ heads) / (d * Nk)
        theta = np.clip(theta, 1e-4, 1.0 - 1e-4)
    return theta, pi, responsibilities, np.array(history)


def observed_coin_loglik(sequences, theta, pi):
    X = np.asarray(sequences, dtype=float)
    n, d = X.shape
    heads = X.sum(axis=1)
    tails = d - heads
    log_prob = np.zeros((n, len(theta)))
    for k in range(len(theta)):
        log_prob[:, k] = np.log(pi[k]) + heads * np.log(theta[k]) + tails * np.log(1.0 - theta[k])
    return float(np.sum(logsumexp(log_prob, axis=1)))


def build_em_ladder():
    d1_sequences = np.array([
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 1, 1, 0],
        [0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    ])
    d2_sequences = rng.binomial(1, np.repeat([0.15, 0.35, 0.65, 0.85], 8)[:, None], size=(32, 12))
    d3_probs = np.repeat([0.42, 0.50, 0.58], 18)
    d3_sequences = rng.binomial(1, d3_probs[:, None], size=(54, 14))
    table = np.array([[8, 2], [7, 3], [6, 4], [4, 6], [3, 7], [2, 8], [5, 5], [9, 1]])
    d4_sequences = np.vstack([np.r_[np.ones(h), np.zeros(t)] for h, t in table])
    high_probs = np.array([0.18, 0.34, 0.52, 0.76])
    d5_sequences = rng.binomial(1, np.repeat(high_probs, 25)[:, None], size=(100, 30))
    return [
        {"name": "D1 two-coin tiny exact latent target", "X": d1_sequences, "K": 2, "reference": 0.0},
        {"name": "D2 four-state mixture", "X": d2_sequences, "K": 4, "reference": 0.0},
        {"name": "D3 overlapping bimodal mixture", "X": d3_sequences, "K": 3, "reference": 0.0},
        {"name": "D4 contingency table with missing labels", "X": d4_sequences, "K": 2, "reference": 0.0},
        {"name": "D5 high-dimensional mixture with bad initialization", "X": d5_sequences, "K": 4, "reference": 0.0},
    ]


## The concept, built once: soft completion then fitting

EM optimizes

$$Q(\theta\mid\theta^{old})=\mathbb E_{z\mid x,\theta^{old}}[\log p(x,z\mid\theta)],\quad \theta^{new}=\arg\max_\theta Q(\theta\mid\theta^{old}).$$

For latent coins, the E-step computes soft component responsibilities and the M-step computes weighted head rates.

In [ ]:

first_heads = 8
first_tails = 2
high = 0.8 ** first_heads * 0.2 ** first_tails
low = 0.3 ** first_heads * 0.7 ** first_tails
responsibility_high = high / (high + low)
print("high-bias responsibility", responsibility_high)
assert responsibility_high > 0.99

sequences = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
])
r = np.array([[0.99, 0.01], [0.05, 0.95]])
heads = sequences.sum(axis=1)
theta0 = np.sum(r[:, 0] * heads) / (10 * np.sum(r[:, 0]))
print("M-step theta", theta0)
assert np.isclose(theta0, (0.99 * 8 + 0.05 * 2) / (10 * (0.99 + 0.05)))


Observed likelihood, not complete-data likelihood under arbitrary labels, is the EM diagnostic guaranteed not to decrease.

In [ ]:

tiny = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
])
old_theta = np.array([0.3, 0.8])
old_pi = np.array([0.5, 0.5])
old_ll = observed_coin_loglik(tiny, old_theta, old_pi)
new_theta, new_pi, resp, hist = em_latent_coin(tiny, K=2, max_iter=1, init_theta=old_theta, init_pi=old_pi)
new_ll = observed_coin_loglik(tiny, new_theta, new_pi)
print("old/new observed log-likelihood", old_ll, new_ll)
assert new_ll >= old_ll - 1e-10


## The dataset ladder

The same Bernoulli-mixture EM handles tiny latent coins, a four-state mixture, overlapping components, contingency rows, and a high-dimensional bad-initialization rung.

In [ ]:

ladder = build_em_ladder()
for i, rung in enumerate(ladder, start=1):
    print(i, rung["name"])
    print("data shape", rung["X"].shape, "components", rung["K"])
    print("row head counts", rung["X"][:5].sum(axis=1))


In [ ]:

rows = []
histories = []
responsibilities = []
for i, rung in enumerate(ladder, start=1):
    K = rung["K"]
    init_theta = np.linspace(0.2, 0.8, K)
    theta, pi, resp, hist = em_latent_coin(rung["X"], K=K, max_iter=35, init_theta=init_theta)
    best_ll = hist[-1]
    gap = float(np.max(hist) - best_ll)
    rows.append((f"D{i}", rung["name"], best_ll, gap))
    histories.append(hist)
    responsibilities.append(resp)

print("rung | observed log-likelihood | final gap")
for label, name, ll, gap in rows:
    print(f"{label} | {name} | {ll:.3f} | {gap:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for i, resp in enumerate(responsibilities):
    ax = axes[0, i]
    ax.imshow(resp[: min(30, len(resp))], aspect="auto", vmin=0.0, vmax=1.0)
    ax.set_title(f"D{i + 1} responsibilities")
    ax.set_xlabel("component")
    ax.set_ylabel("row")

ax = axes[1, 0]
for i, hist in enumerate(histories):
    normalized = hist - hist[0]
    ax.plot(normalized, label=f"D{i + 1}")
ax.set_title("log-likelihood gain vs iteration")
ax.set_xlabel("iteration")
ax.set_ylabel("gain")
ax.legend(fontsize=8)
for j in range(1, 5):
    axes[1, j].axis("off")
plt.tight_layout()


## Pitfall on D5: hardening and local optima

A bad initialization or hard E-step can trap EM. We reproduce a weak local solution, then fix it with soft responsibilities and multiple starts.

In [ ]:

d5 = ladder[-1]
bad_theta, bad_pi, bad_resp, bad_hist = em_latent_coin(d5["X"], K=4, max_iter=20, init_theta=np.array([0.45, 0.48, 0.51, 0.54]))
best_ll = -np.inf
best_hist = None
for start in range(5):
    init_theta = np.sort(rng.uniform(0.1, 0.9, size=4))
    theta, pi, resp, hist = em_latent_coin(d5["X"], K=4, max_iter=40, init_theta=init_theta)
    if hist[-1] > best_ll:
        best_ll = hist[-1]
        best_hist = hist

print("bad initialization ll", bad_hist[-1])
print("best multi-start ll", best_ll)
assert best_ll >= bad_hist[-1]


## Evaluate it + practice

- Metric: observed log-likelihood gap or responsibility error versus a reference run.
- No-skill baseline: assign rows to components by hard majority head count once and never update softly.
- Cheap sanity check: observed log-likelihood should not decrease after each EM update.
- Ablation: harden responsibilities before the M-step and the D5 likelihood drops or stalls.
- Failure signals: component collapse, decreasing observed likelihood, or very different solutions from nearby starts.

**Practice.** Run more random starts on D5 and plot the distribution of final log-likelihoods.

**Practice.** Add a Dirichlet-like pseudo-count to pi and inspect component collapse.

**Practice.** Create overlapping D3 components and compare soft and hard assignments.